# 📦 Template 1 : préparation des données — AT&T Spam Detector

*Préparé par Claude le 19/10/2026 pour le projet `01-AT&T_spam_detector.ipynb`. Aucun code de solution ici : c'est toi qui écris.*

## Les 4 templates du projet

| # | Phase | Entrée | Sortie |
|---|---|---|---|
| **1** | **Préparation des données** (ce notebook) | `src/spam.csv` | `src/train.csv`, `src/val.csv`, `src/test.csv` |
| 2 | Modèle simple : embeddings + petit réseau (M05-D05) | les 3 CSV | un modèle + ses scores |
| 3 | Transfer learning : modèle Hugging Face (M06-D02) | les 3 CSV | un modèle + ses scores |
| 4 | Évaluation et conclusion | les scores | le tableau final du livrable |

Les CSV de sortie servent de point de départ aux templates 2 et 3 : ce notebook-ci n'est à faire qu'**une fois**.

## Comment s'en servir

Chaque étape suit le même rythme :

1. **🎯 Objectif** : ce qu'on veut obtenir, et pourquoi ;
2. **🧰 Outils** : les fonctions utiles, leur rôle, leurs arguments importants ;
3. **✍️ À toi** : une cellule avec le plan en commentaires, que tu remplis ;
4. **✅ Contrôle** : une cellule déjà écrite qui vérifie tes chiffres. Lance-la : ✅ = c'est bon, `AssertionError` = relis l'étape.

Les noms de variables imposés (`df`, `df_clean`, `train_df`...) sont ceux qu'attendent les contrôles.

---
# Étape 0 — Les imports
Rien à écrire : lance la cellule.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split

DOSSIER_SRC = Path("src")       # le dossier où se trouve spam.csv (à côté de ce notebook)
SEED = 42                       # graine du hasard, la même partout
print("prêt")

---
# Étape 1 — Charger le fichier

🎯 Lire `src/spam.csv` dans un DataFrame `df`.

⚠️ Le fichier n'est **pas** en UTF-8 (l'encodage par défaut de pandas) : sans le bon encodage, `read_csv` plante avec `UnicodeDecodeError`. Il est en **latin-1**.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `pd.read_csv(chemin, encoding=...)` | lit un CSV dans un DataFrame | `encoding="latin-1"` |
| `.head()`, `.shape`, `.info()` | premier coup d'œil | — |

In [ ]:
# ✍️ À toi
# 1. df = le fichier DOSSIER_SRC / "spam.csv" lu avec le bon encodage
# 2. affiche les 5 premières lignes, la forme, puis .info()

👀 **Observe** : combien de colonnes ? Que contiennent `v1` et `v2` ? Combien de valeurs non vides dans les 3 colonnes `Unnamed` ?

In [ ]:
# ✅ Contrôle
assert df.shape == (5572, 5), f"forme attendue (5572, 5), obtenue {df.shape}"
assert list(df.columns[:2]) == ["v1", "v2"]
print("✅ étape 1")

---
# Étape 2 — Nettoyer

## 2.a Repérer les messages coupés

🎯 Les colonnes `Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4` sont presque vides. Mais quand elles ne le sont pas, ce n'est **pas** du bruit : c'est **la fin d'un SMS**. Certains messages contiennent des guillemets mal fermés, et le CSV les a coupés à chaque virgule.

```
v1     v2                                   Unnamed: 2       Unnamed: 3
spam   "Your free ringtone ... FML"          " PO Box 5249"   " MK17 92H. 450Ppw 16\""
       └─────────── un seul SMS, coupé en 3 morceaux à chaque virgule ───────────┘
```

Si tu supprimes ces colonnes tout de suite, tu perds la fin de ces messages.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `df[[col1, col2, col3]]` | garder plusieurs colonnes (double crochet = liste) | — |
| `.notna()` | `True` là où il y a une valeur | — |
| `.any(axis=1)` | `True` si **au moins une** case de la **ligne** est `True` | `axis=1` = ligne par ligne |
| `.sum()` | compte les `True` | — |

In [ ]:
# ✍️ À toi
# 1. cols_en_trop = la liste des 3 noms de colonnes "Unnamed"
# 2. masque_coupes = True pour chaque ligne qui a au moins une valeur dans ces 3 colonnes
# 3. affiche le nombre de messages coupés, puis 3 de ces lignes pour les regarder

In [ ]:
# ✅ Contrôle
assert masque_coupes.sum() == 50, f"50 messages coupés attendus, obtenu {masque_coupes.sum()}"
print("✅ étape 2.a")

## 2.b Recoller les morceaux

🎯 Créer une colonne `text` = `v2` + les morceaux des colonnes `Unnamed`, **recollés avec une virgule** (la virgule que le CSV a prise pour un séparateur). Pour les 5 522 messages entiers, `text` = `v2`.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `",".join(morceaux)` | colle des textes avec une virgule entre eux | les morceaux doivent être des **str** (pas de `NaN`) |
| `.dropna()` sur **une ligne** | enlève les cases vides de cette ligne | — |
| `.apply(fonction, axis=1)` | applique une fonction à **chaque ligne** | `axis=1` |

Démo des deux premiers outils **sur une seule ligne**, dans un autre contexte (une adresse coupée) :

In [ ]:
adresse = pd.Series(["12 rue des Lilas", " Nantes", None, None])   # une ligne avec des cases vides
print(adresse.dropna().tolist())          # les cases vides ont disparu
print(",".join(adresse.dropna()))         # recollé avec une virgule

Chez toi, c'est pareil, mais pour les 5 572 lignes : il faut appliquer ce geste à chaque ligne du DataFrame.

In [ ]:
# ✍️ À toi
# 1. cols_texte = "v2" + les 3 colonnes Unnamed (dans cet ordre)
# 2. df["text"] = pour chaque ligne de df[cols_texte] : enlever les cases vides, recoller avec ","
# 3. affiche le text d'une ligne de masque_coupes pour vérifier

In [ ]:
# ✅ Contrôle
exemple = df.loc[df["v2"].str.startswith("Your free ringtone is waiting"), "text"].iloc[0]
assert exemple.endswith('MK17 92H. 450Ppw 16"'), f"la fin du message manque : {exemple!r}"
assert (df.loc[~masque_coupes, "text"] == df.loc[~masque_coupes, "v2"]).all(), "les messages entiers doivent rester identiques à v2"
print("✅ étape 2.b")

## 2.c Garder deux colonnes propres

🎯 Fabriquer `df_clean` avec **deux colonnes seulement** :
- `text` : le SMS recollé ;
- `label` : **0** pour `ham`, **1** pour `spam` (le modèle veut des nombres ; 1 = la classe qu'on cherche).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.map({ancienne: nouvelle, ...})` | remplace chaque valeur via un dictionnaire | les clés doivent couvrir **toutes** les valeurs, sinon `NaN` |
| `df[[...]].copy()` | nouveau DataFrame indépendant de `df` | — |

In [ ]:
# ✍️ À toi
# 1. df["label"] = v1 traduit en 0 / 1
# 2. df_clean = une copie de df avec seulement les colonnes text et label (dans cet ordre)
# 3. affiche df_clean.head() et la répartition de label

In [ ]:
# ✅ Contrôle
assert list(df_clean.columns) == ["text", "label"], f"colonnes obtenues : {list(df_clean.columns)}"
assert df_clean["label"].isna().sum() == 0, "des NaN dans label : la traduction a raté une valeur"
assert df_clean["label"].sum() == 747, f"747 spams attendus avant dédoublonnage, obtenu {df_clean['label'].sum()}"
print("✅ étape 2.c")

## 2.d Supprimer les doublons

🎯 Le même SMS apparaît plusieurs fois (spams envoyés en masse, « Ok lar... »). On garde **un exemplaire** de chaque texte.

❓ Question du matin : pourquoi le faire **avant** de séparer train et test ? Pense à ce qui arrive si un même SMS tombe dans les deux : le modèle passe l'examen avec une question qu'il a déjà vue en révision.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.duplicated(subset=...)` | `True` pour chaque ligne déjà vue plus haut | `subset="text"` : on compare seulement le texte |
| `.drop_duplicates(subset=...)` | garde la première, supprime les suivantes | `subset="text"` |
| `.reset_index(drop=True)` | renumérote 0, 1, 2... | `drop=True` : jette l'ancien index |

In [ ]:
# ✍️ À toi
# 1. affiche le nombre de doublons sur la colonne text
# 2. df_clean = df_clean sans doublons de text, index renuméroté
# 3. affiche la nouvelle forme et le nombre de spams

In [ ]:
# ✅ Contrôle
assert df_clean.shape == (5169, 2), f"forme attendue (5169, 2), obtenue {df_clean.shape}"
assert df_clean["label"].sum() == 653, f"653 spams attendus, obtenu {df_clean['label'].sum()}"
assert df_clean.index.tolist() == list(range(5169)), "index pas renuméroté"
print("✅ étape 2.d")

---
# Étape 3 — Explorer

## 3.a Répartition des classes

🎯 Quelle part de spams ? C'est elle qui dit si l'accuracy est une bonne métrique ici.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.value_counts(normalize=True)` | part de chaque valeur (entre 0 et 1) | `normalize=True` |
| `px.histogram(df, x=...)` | diagramme en barres d'une colonne | `x="label"` |

In [ ]:
# ✍️ À toi
# 1. affiche la part de ham et de spam
# 2. trace un diagramme en barres de label

In [ ]:
# ✅ Contrôle
assert round(df_clean["label"].mean(), 4) == 0.1263
print("✅ étape 3.a")

✍️ **À écrire (une phrase)** : un « modèle » qui répond **toujours ham** aurait quelle accuracy ? Qu'est-ce que ça dit de l'accuracy comme métrique pour ce projet ?

*Ta réponse :*

## 3.b Longueur des messages

🎯 Comparer la longueur des ham et des spams, en caractères et en mots. Ça servira aussi en phase 2 pour choisir la longueur des séquences.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.str.len()` | nombre de caractères de chaque texte | — |
| `.str.split().str.len()` | nombre de mots (on découpe aux espaces, puis on compte les morceaux) | — |
| `.groupby("label")[cols].median()` | médiane par classe | — |
| `px.histogram(df, x=..., color=...)` | deux histogrammes superposés | `color="label"`, `barmode="overlay"` |

In [ ]:
# ✍️ À toi
# 1. df_clean["n_car"] = nombre de caractères ; df_clean["n_mots"] = nombre de mots
# 2. affiche la médiane de n_car et n_mots pour chaque classe
# 3. histogramme de n_mots, une couleur par classe

In [ ]:
# ✅ Contrôle
med = df_clean.groupby("label")[["n_car", "n_mots"]].median()
assert med.loc[0, "n_car"] == 53 and med.loc[1, "n_car"] == 149, med
assert med.loc[0, "n_mots"] == 11 and med.loc[1, "n_mots"] == 25, med
print("✅ étape 3.b")

## 3.c Combien de mots garder par message ? (préparation de la phase 2)

🎯 En phase 2, chaque SMS deviendra une suite de numéros de **même longueur** (on coupe les trop longs, on complète les trop courts). Choisir une longueur `max_len` qui couvre presque tous les messages, sans s'aligner sur le plus long.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.max()` | le plus long | — |
| `.quantile(q)` | la valeur sous laquelle se trouvent `q` % des messages | `q=0.95` |

In [ ]:
# ✍️ À toi
# 1. affiche le nombre de mots du message le plus long
# 2. affiche le 95e centile (quantile 0.95) de n_mots

In [ ]:
# ✅ Contrôle
assert df_clean["n_mots"].max() == 171
assert round(df_clean["n_mots"].quantile(0.95)) == 33
print("✅ étape 3.c")

✍️ **À écrire** : quelle valeur de `max_len` choisis-tu, et pourquoi pas 171 ?

*Ta réponse :*

## 3.d Lire les messages

🎯 Lire de vrais exemples : les chiffres ne disent pas tout. Repère ce qui distingue un spam (mots, chiffres, majuscules, numéros courts...).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `df[df["label"] == 1]` | garder seulement les spams | — |
| `.sample(n, random_state=...)` | n lignes au hasard, reproductibles | `random_state=SEED` |
| `.str.lower().str.split().explode().value_counts()` | les mots les plus fréquents (en minuscules, un mot par ligne, puis compter) | `.head(20)` |
| `.str.contains(motif).sum()` | combien de textes contiennent un motif | — |

In [ ]:
# ✍️ À toi
# 1. affiche 5 ham et 5 spams au hasard (texte complet : pd.set_option("display.max_colwidth", None))
# 2. affiche les 20 mots les plus fréquents des spams, puis ceux des ham
# 3. compte les messages qui contiennent "&lt;" : que remplace ce code, et dans quelle classe apparaît-il ?

✍️ **À écrire (2-3 phrases)** : ce qui distingue un spam d'après tes lectures, et ce que tu fais de `&lt;#&gt;` (le garder tel quel, le remplacer ?).

*Ta réponse :*

---
# Étape 4 — Séparer train / validation / test

🎯 Trois paquets, avec **la même part de spams (12,63 %)** dans chacun :

```
df_clean : 5 169 SMS
┌──────────────────── 70 % train ────────────────────┬── 15 % val ──┬── 15 % test ──┐
│ le modèle apprend dessus                            │ on règle :   │ on ne le      │
│                                                     │ époques,     │ regarde qu'à  │
│                                                     │ seuil...     │ la toute fin  │
└─────────────────────────────────────────────────────┴──────────────┴───────────────┘
```

`train_test_split` ne coupe qu'en **deux** : il faut l'appeler **deux fois**.
1. couper `df_clean` en `train_val_df` (85 %) et `test_df` (15 %) ;
2. couper `train_val_df` en `train_df` et `val_df`. Attention : 15 % du total = **15 / 85** de `train_val_df`.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `train_test_split(df, ...)` | coupe un DataFrame en deux | `test_size=`, `stratify=` la colonne des classes, `random_state=SEED` |

Les colonnes `n_car` et `n_mots` peuvent partir avec : elles ne gênent pas.

In [ ]:
# ✍️ À toi
# 1. train_val_df, test_df = 1re coupe (test = 15 %), stratifiée sur label
# 2. train_df, val_df = 2e coupe de train_val_df (val = 15 / 85), stratifiée sur label
# 3. affiche la taille et la part de spams de chaque paquet

In [ ]:
# ✅ Contrôle
assert (len(train_df), len(val_df), len(test_df)) == (3617, 776, 776), (len(train_df), len(val_df), len(test_df))
for nom, paquet in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert round(paquet["label"].mean(), 4) == 0.1263, f"{nom} : stratification ratée"
assert not set(train_df["text"]) & set(test_df["text"]), "un même SMS est dans train ET test"
print("✅ étape 4")

---
# Étape 5 — Enregistrer pour les phases suivantes

🎯 Écrire les trois paquets dans `src/train.csv`, `src/val.csv`, `src/test.csv`, avec seulement `text` et `label`. Les templates 2 et 3 partiront de ces fichiers.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `.to_csv(chemin, index=False)` | écrit un CSV | `index=False` : sinon une colonne de numéros en plus |

Ces fichiers sont écrits en UTF-8 (le défaut de `to_csv`) : plus besoin de `encoding=` pour les relire.

In [ ]:
# ✍️ À toi
# Pour chacun des 3 paquets : garder text et label, écrire dans DOSSIER_SRC / "<nom>.csv"

In [ ]:
# ✅ Contrôle
for nom, n in [("train", 3617), ("val", 776), ("test", 776)]:
    relu = pd.read_csv(DOSSIER_SRC / f"{nom}.csv")
    assert list(relu.columns) == ["text", "label"], f"{nom}.csv : colonnes {list(relu.columns)}"
    assert len(relu) == n, f"{nom}.csv : {len(relu)} lignes au lieu de {n}"
print("✅ étape 5 : données prêtes pour les templates 2 et 3")

---
# 📝 Pour le livrable

Le notebook final doit **expliquer** le preprocessing. Garde de côté, en une ligne chacun :

| Décision | Ta justification |
|---|---|
| encodage latin-1 | |
| 50 messages recollés | |
| 403 doublons supprimés avant le split | |
| métrique principale (pas l'accuracy) | |
| `max_len` choisi | |
| 70 / 15 / 15 stratifié | |
| `&lt;#&gt;` | |